# Laguna State Polytechnic University
## College of Computer Studies
### CSST 102 – Basic Machine Learning | Final Project
---
# AniTaya: Philippine Crop Recommendation, Harvest Prediction & Income Estimation System

**Authoritative Philippine Datasets (`data/philippines_crop_yield_climate.csv`):**
- **Crop Production & Harvested Area:** Philippine Statistics Authority (PSA OpenSTAT `DB/2E/CS`, 1987–2025, 82 provinces × 35 crops)
- **Provincial Farmgate Prices & Palay Costs:** PSA OpenSTAT `DB/2M/NFG` & Table 5.5 (`DB/3S/C5`)
- **Climate Normals & Time Series:** DOST-PAGASA CliMap v2.1 (1991–2020 GeoTIFF Normals) & NASA POWER MERRA-2 (1987–2025)

**Target Variable:** `value_hg_ha` (Crop harvest yield in hectograms per hectare = `(production_mt / harvested_area_ha) * 10000`)  
**Inference Features (4 Reliable Inputs Only):** `item` (Crop), `area` (Philippine Province), `average_rain_fall_mm_per_year`, `avg_temp`  
**5 Core Evaluated CSST 102 Algorithms:** Linear Regression (Baseline), Ridge Regression, Decision Tree Regressor, Random Forest Regressor, Gradient Boosting Regressor

## 1. Import Libraries & Load Master Philippine Dataset

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, GroupKFold, cross_validate
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

data_path = os.path.join("..", "data", "philippines_crop_yield_climate.csv")
df = pd.read_csv(data_path)
print(f"Full Philippine Archive (1987-2025): {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Provinces: {df['area'].nunique()} | Crops: {df['item'].nunique()}")
df[["area", "year", "item", "average_rain_fall_mm_per_year", "avg_temp", "yield_mt_ha", "value_hg_ha"]].head(8)

## 2. Exploratory Data Analysis (EDA) & Data Integrity Checks

In [ ]:
feature_cols = ["item", "average_rain_fall_mm_per_year", "avg_temp", "area"]
target_col = "value_hg_ha"

print("=== MISSING VALUES CHECK ===")
print(df[feature_cols + [target_col, "year"]].isnull().sum())

dups = df.duplicated(subset=["area", "year", "item"]).sum()
print(f"\nExact (province, year, crop) duplicates: {dups}")

print("\n=== TOP CROPS BY OBSERVATION COUNT ===")
print(df["item"].value_counts().head(12))

## 3. Homogeneous Era Selection (`2010–2025`), Chronological Split & Leak-Free Preprocessing

- **Why `year >= 2010` for ML training?** From 1987–2009 only `Rice, paddy` and `Maize` exist in PSA OpenSTAT tables, whereas all other 33 PSA crops begin at `2010`. Using the uniform `2010–2025` panel (`32,618` rows) prevents legacy 1990s rice regimes from biasing modern multi-crop yield forecasts while preserving the full `36,088`-row archive.
- **Chronological Split:** Train/Validation = `2010–2021` (`24,369` rows); Unseen Holdout Test = `2022–2025` (`8,249` rows). All scalers & encoders fit strictly inside training folds.

In [ ]:
df_ml = df[df["year"] >= 2010].copy().reset_index(drop=True)
train_df = df_ml[df_ml["year"] <= 2021].copy().reset_index(drop=True)
test_df = df_ml[df_ml["year"] >= 2022].copy().reset_index(drop=True)

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

print(f"Train Split (2010-2021): {len(train_df):,} rows ({train_df['item'].nunique()} crops)")
print(f"Holdout Test (2022-2025): {len(test_df):,} rows ({test_df['item'].nunique()} crops)")

categorical_features = ["item", "area"]
numeric_features = ["average_rain_fall_mm_per_year", "avg_temp"]

def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
            ("num", StandardScaler(), numeric_features),
        ]
    )

## 4. 5-Model Training, 5-Fold Cross-Validation & Chronological Evaluation
Exact mirror of `../train_model.py`. Model selection depends strictly on **5-Fold Validation CV $R^2$** inside the `2010–2021` training split (zero holdout test peeking).

In [ ]:
candidates = {
    "Linear Regression (Baseline)": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=24, min_samples_leaf=5, random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, max_depth=None, min_samples_leaf=2, random_state=42, n_jobs=2),
    "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=220, max_depth=6, learning_rate=0.08, random_state=42),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
rows = []
trained = {}

for name, est in candidates.items():
    pipe = Pipeline([("pre", make_preprocessor()), ("reg", est)])
    cv = cross_validate(pipe, X_train, y_train, cv=kf, scoring=("r2", "neg_root_mean_squared_error"), n_jobs=1)
    pipe.fit(X_train, y_train)
    p_te = pipe.predict(X_test)
    rows.append({
        "Algorithm": name,
        "5-Fold CV R2": round(float(np.mean(cv["test_r2"])), 4),
        "Holdout Test R2": round(float(r2_score(y_test, p_te)), 4),
        "Test MAE (hg/ha)": round(float(mean_absolute_error(y_test, p_te)), 2),
        "Test RMSE (hg/ha)": round(float(root_mean_squared_error(y_test, p_te)), 2),
    })
    trained[name] = pipe

results_df = pd.DataFrame(rows).sort_values("5-Fold CV R2", ascending=False).reset_index(drop=True)
results_df

## 5. Visualize Model Comparison & Feature Importance

In [ ]:
plt.figure(figsize=(9, 4), dpi=120)
plt.barh(results_df["Algorithm"][::-1], results_df["5-Fold CV R2"][::-1], color="#0d9488")
plt.xlim(0, 1.05)
plt.xlabel("5-Fold Cross-Validation R2 Score")
plt.title("AniTaya PH Dataset: 5-Model CV Performance")
plt.tight_layout()
plt.show()

best_pipe = trained["Random Forest Regressor"]
rf = best_pipe.named_steps["reg"]
pre = best_pipe.named_steps["pre"]
feat_names = list(pre.named_transformers_["cat"].get_feature_names_out(categorical_features)) + numeric_features

agg = {"Crop Type (item)": 0.0, "Philippine Province (area)": 0.0, "Annual Rainfall": 0.0, "Average Temp": 0.0}
for fn, imp in zip(feat_names, rf.feature_importances_):
    if fn.startswith("item_"):
        agg["Crop Type (item)"] += imp
    elif fn.startswith("area_"):
        agg["Philippine Province (area)"] += imp
    elif fn == "average_rain_fall_mm_per_year":
        agg["Annual Rainfall"] += imp
    elif fn == "avg_temp":
        agg["Average Temp"] += imp

fi = pd.DataFrame([(k, round(v * 100, 2)) for k, v in agg.items()], columns=["Feature", "Importance_%"]).sort_values("Importance_%")
plt.figure(figsize=(8, 3.8), dpi=120)
plt.barh(fi["Feature"], fi["Importance_%"], color="#10b981")
plt.xlabel("Relative Importance (%)")
plt.title("Random Forest Feature Importance (Philippine Features Only)")
plt.tight_layout()
plt.show()

## 6. Sample End-to-End Philippine Prediction (Production Artifact)

In [ ]:
# Load deployed pipeline saved by ../train_model.py
prod_pipe = joblib.load(os.path.join("..", "models", "anitaya_best_pipeline.joblib"))
sample = pd.DataFrame([{
    "item": "Rice, paddy",
    "average_rain_fall_mm_per_year": 2050.0,
    "avg_temp": 27.4,
    "area": "Laguna",
}])

pred_hg_ha = float(prod_pipe.predict(sample)[0])
yield_mt_ha = pred_hg_ha / 10000.0
land_area = 2.0
total_mt = yield_mt_ha * land_area
total_kg = total_mt * 1000.0

price_php_kg = 17.38    # PSA OpenSTAT 2025 median reference
cost_per_ha = 59695.0   # PSA Table 5.5 2024 All Palay Average Total Costs
gross = total_kg * price_php_kg
cost = cost_per_ha * land_area
net = gross - cost

print(f"Predicted Yield Rate: {yield_mt_ha:.2f} MT/ha")
print(f"Total Harvest (2.0 ha): {total_mt:.2f} MT ({total_kg:,.0f} kg / {total_kg/50:,.1f} cavans)")
print(f"Est. Gross Revenue: PHP {gross:,.2f} | Cost: PHP {cost:,.2f} | Net: PHP {net:,.2f}")